In [ ]:
### Load libraries and set params
library(Matrix)
library(parallel)
library(Seurat)
library(gridExtra)
library(ggplot2)
library(future)
library(ggpubr)
library(viridis)
library(openxlsx)
library(viridis)
library(scran)
library(gghighlight)
library(dplyr)
library(org.Dm.eg.db)
library(ggExtra)
library(scDblFinder)
library(patchwork)
library(RhpcBLASctl)
blas_set_num_threads(18)
library(peakRAM)
options(device=pdf)
options(future.globals.maxSize = 214748364800)
library(future)
plan("multicore", workers = 18)
library(SeuratDisk)
library(reshape2)
library(tidyverse)
library(RColorBrewer)
library(zellkonverter)

### Set directories
mainDir <- "/data/ebaird/scRNAseq/SCENTINELsep24/"
repDir <- paste0(mainDir, "label_transfer/")
figDir <- paste0(repDir, "figs/")
tabDir <- paste0(repDir, "tables/")
refsDir <- "/data/ebaird/scRNAseq/refs/"


dir.create(repDir, recursive = TRUE, showWarnings = FALSE)
dir.create(figDir, recursive = TRUE, showWarnings = FALSE)
dir.create(tabDir, recursive = TRUE, showWarnings = FALSE)
dir.create(refsDir, recursive = TRUE, showWarnings = FALSE)

### Set colours
mycols <- c(1, '#ffffe5','#fff7bc','#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506')
mycols11 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray")
mycols13 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green")
mycols17 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green", rainbow(4))

mycols20 <- c("yellow", '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "chartreuse", "blue", "green", rainbow(4), "darkslategray3", "darksalmon", "darkorchid4", "cyan", "magenta")

corner <- function(x) x[1:5,1:5]
cols <- c(colorRamps::matlab.like2(20)[1:18], "deeppink2", "deeppink3", "deeppink4")

getdensity <- function(x, y, ...) {
      dens <- MASS::kde2d(x, y, ...)
      ix <- findInterval(x, dens$x)
      iy <- findInterval(y, dens$y)
      ii <- cbind(ix, iy)
      return(dens$z[ii])
}

best_colors <- viridis(20)
best_colors_magma <- magma(20)
best_colors_plasma <- plasma(20)

In [ ]:
# Load seurat objects

seu_reference <- readRDS("/data/ebaird/scRNAseq/SCENTINELsep25/QC_clustering/final_clusters.rds")
seu_query <- readRDS("/data/ebaird/scRNAseq/SCENTINELsep24/QC_clustering/merged_clusters.rds")

In [ ]:
DimPlot(seu_query, label = TRUE, reduction = 'umap')
jpeg(paste0(figDir, "Dillon_progenitor_atlas_umap.jpg"), width = 2000, height = 2000, res = 150)
DimPlot(seu_query, label = TRUE, reduction = 'umap')
dev.off()

In [ ]:
seu_query <- UpdateSeuratObject(seu_query)

In [ ]:
seu_query <- readH5AD("/data/ebaird/scRNAseq/public_datasets/Michki/GSE153723_typeII_OL_merged.h5ad")
assayNames(seu_query) <- "counts"
assay(seu_query, "logcounts") <- log1p(assay(seu_query, "counts"))
seu_query <- as.Seurat(seu_query)

In [ ]:
seu_query
seu_reference

In [ ]:
# Check active assays
DefaultAssay(seu_reference)
DefaultAssay(seu_query)

# They should both be "RNA"
DefaultAssay(seu_reference) <- "RNA"
DefaultAssay(seu_query) <- "RNA"

# How many shared genes?
length(intersect(rownames(seu_reference), rownames(seu_query)))


In [ ]:
shared.features <- intersect(
  VariableFeatures(seu_reference),
  VariableFeatures(seu_query)
)

length(shared.features)


In [ ]:
shared.features <- intersect(
  rownames(seu_reference),
  rownames(seu_query)
)

length(shared.features)

In [ ]:
VariableFeatures(seu_reference) <- shared.features

In [ ]:
seu_reference <- ScaleData(seu_reference, features = shared.features, verbose = FALSE)
seu_reference <- RunPCA(seu_reference, features = shared.features, npcs = 50, verbose = FALSE)

seu_query <- ScaleData(seu_query, features = shared.features, verbose = FALSE)
seu_query <- RunPCA(seu_query, features = shared.features, npcs = 50, verbose = FALSE)

In [ ]:
anchors <- FindTransferAnchors(
  reference = seu_reference,
  query = seu_query,
  features = shared.features,
  dims = 1:30,
  reference.reduction = "pca"
)

In [ ]:
colnames(seu_reference@meta.data)

In [ ]:
unique(seu_reference$seurat_clusters)

In [ ]:
ElbowPlot(seu_reference)

In [ ]:
# re-run clustering only
seu_reference <- FindNeighbors(seu_reference, dims = 1:20, verbose = FALSE)
seu_reference <- FindClusters(seu_reference, resolution = 1, verbose = FALSE)

In [ ]:
predictions <- TransferData(
  anchorset = anchors,
  refdata = seu_reference$seurat_clusters,
  dims = 1:30
)

seu_query <- AddMetaData(seu_query, metadata = predictions)

In [ ]:
seu_query@reductions

In [ ]:
plots <- list()
unique_ids <- sort(unique(seu_query$predicted.id))

for (id in unique_ids) {

  # Highlight cells in query
  cells_to_highlight <- WhichCells(seu_query, expression = predicted.id == id)
  p1 <- DimPlot(seu_query, group.by = "predicted.id", reduction = "umap",
                cells.highlight = cells_to_highlight,
                cols.highlight = "red", 
                cols = "grey80",
                pt.size = 0.2,           # size of non-highlighted cells
                sizes.highlight = 0.2) +   # size of highlighted cells
        ggtitle(paste("2024:", id))

  # Highlight cells in reference
  cells_to_highlight <- WhichCells(seu_reference, expression = seurat_clusters == id)
  p2 <- DimPlot(seu_reference, group.by = "seurat_clusters", reduction = "umap",
                cells.highlight = cells_to_highlight,
                cols.highlight = "red", 
                cols = "grey80",
                pt.size = 0.2,
                sizes.highlight = 0.2) +
        ggtitle(paste("2025:", id))

  plots[[id]] <- p1 / p2
}

final_plot <- wrap_plots(plots)

jpeg(paste0(figDir, "UMAP_label_transfer_2025sc_to_2024sc.jpg"),
     width = 3500, height = 5000, res = 150)
final_plot
dev.off()

In [ ]:
# Reference based integration

anchors <- FindTransferAnchors(
  reference = seu_reference,
  query = seu_query,
  dims = 1:30,
  reference.reduction = "pca"
)

In [ ]:
library(future)
library(future.apply)

plan(multicore, workers = 48)
options(future.globals.maxSize = 200 * 1024^3)
plan(multisession, workers = 48)

options(Seurat.object.assay.version = "v5")
options(Seurat.parallel = TRUE)


In [ ]:
# Re-run umap to return model
seu_reference <- RunUMAP(seu_reference, dims = 1:30, verbose = TRUE, return.model = TRUE, reduction_name = "umap")

In [ ]:
tumor_mapped <- MapQuery(
  anchorset = anchors,
  query = seu_query,
  reference = seu_reference,
  # refdata = list(celltype = "celltype"), # optional if WT annotated
  reference.reduction = "pca",
  reduction.model = "umap",
  verbose = TRUE
)